# 02 · 复现外部测试集结果

加载 notebook 01 训练好的 AMP-BERT,在外部测试集(DRAMP 的 AMP ∪ AMPEP 的 non-AMP)上评估。

> 需先运行 notebook 01。若模型存在 Google Drive,记得挂载并把 `MODEL_DIR` 指过去。

## 0 · 环境准备
安装一个仍带经典 BERT 分词器的 `transformers` 版本(Colab 自带的太新,无法正确加载 ProtBERT-BFD),并检查 GPU。

> 运行前请先设置:**代码执行程序 → 更改运行时类型 → GPU**。

In [ ]:
%pip install -q transformers==4.46.3 "tokenizers>=0.20,<0.21" sentencepiece

In [ ]:
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('使用设备:', device)   # 期望输出 cuda

## 1 · 导入依赖

In [ ]:
import os
import numpy as np
import pandas as pd
from torch.utils.data import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          Trainer, TrainingArguments)
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             matthews_corrcoef, roc_auc_score)

In [ ]:
# 整个 notebook 共用的两个常量
MODEL_NAME = 'Rostlab/prot_bert_bfd'   # 预训练模型(分词器与模型都用它)
MAX_LEN    = 200                        # 序列截断/补齐到的长度

## 2 · 准备外部测试集
把 AMP 与 non-AMP 两个文件合并成完整测试集。

In [ ]:
AMP_URL    = 'https://raw.githubusercontent.com/BioGavin/AMP-BERT/main/data/raw/veltri_dramp_cdhit_90.csv'
NONAMP_URL = 'https://raw.githubusercontent.com/BioGavin/AMP-BERT/main/data/raw/non_amp_ampep_cdhit90.csv'

amp_df    = pd.read_csv(AMP_URL, index_col=0)
nonamp_df = pd.read_csv(NONAMP_URL, index_col=0)
test_df = pd.concat([amp_df, nonamp_df])
print('AMP:', len(amp_df), '| non-AMP:', len(nonamp_df), '| 合计:', len(test_df))
test_df.head()

## 3 · 把序列编码成模型输入

In [ ]:
class AmpDataset(Dataset):
    """把一个 DataFrame(列:aa_seq, AMP)编码成模型输入。"""
    def __init__(self, df, tokenizer, max_len=MAX_LEN):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.seqs = list(self.df['aa_seq'])
        self.labels = list(self.df['AMP'].astype(int))

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        # ProtBERT 要求氨基酸之间用空格分隔
        seq = ' '.join(''.join(self.seqs[idx].split()))
        ids = self.tokenizer(seq, truncation=True, padding='max_length',
                             max_length=self.max_len)
        sample = {k: torch.tensor(v) for k, v in ids.items()}
        sample['labels'] = torch.tensor(self.labels[idx])
        return sample

In [ ]:
# 加载分词器(use_fast=False 直接用 vocab.txt 的 WordPiece 分词器,避开新版本的兼容问题)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, do_lower_case=False, use_fast=False)

In [ ]:
test_dataset = AmpDataset(test_df, tokenizer)
print('test_dataset 大小:', len(test_dataset))

## 4 · 定义评估指标

In [ ]:
def compute_metrics(pred):
    labels = pred.label_ids
    logits = pred.predictions
    preds = logits.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average='binary', zero_division=0)
    m = {'accuracy': accuracy_score(labels, preds), 'f1': f1,
         'precision': precision, 'recall': recall,
         'mcc': matthews_corrcoef(labels, preds)}
    if len(np.unique(labels)) == 2:                 # AUC 需要正类概率
        z = logits - logits.max(axis=-1, keepdims=True)
        probs = np.exp(z) / np.exp(z).sum(axis=-1, keepdims=True)
        m['roc_auc'] = roc_auc_score(labels, probs[:, 1])
    return m

## 5 · 加载训练好的 AMP-BERT

In [ ]:
MODEL_DIR = 'models/amp_bert'   # 来自 notebook 01
assert os.path.exists(MODEL_DIR), f'{MODEL_DIR} 不存在,请先运行 notebook 01。'
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)

## 6 · 评估

In [ ]:
eval_args = TrainingArguments(output_dir='results/amp_bert_test',
                              per_device_eval_batch_size=56, report_to='none')
trainer = Trainer(model=model, args=eval_args, compute_metrics=compute_metrics)
predictions, label_ids, metrics = trainer.predict(test_dataset)
metrics

## 7 · 保存逐条预测结果

In [ ]:
PRED_CSV = 'results/external_test_predictions.csv'
os.makedirs('results', exist_ok=True)
out = test_df.copy()
out['pred'] = predictions.argmax(-1)
out.to_csv(PRED_CSV)
print('已写入', PRED_CSV)